In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

In [28]:
train = pd.read_csv('./result/상위5개컬럼모음.csv')
# test = pd.read_csv('상위6개컬럼모음.test.csv')

In [29]:
X_train=train.drop(['Segment'],axis=1)
y=train['Segment']
# X_test = test

In [35]:
print(y.value_counts().sort_index())

Segment
A        972
B        144
C     127590
D     349242
E    1922052
Name: count, dtype: int64


In [4]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400000 entries, 0 to 2399999
Data columns (total 5 columns):
 #   Column         Dtype
---  ------         -----
 0   _2순위카드이용금액     int64
 1   쇼핑_도소매_이용금액    int64
 2   _1순위교통업종_이용금액  int64
 3   연체입금원금_B0M     int64
 4   잔액_일시불_B0M     int64
dtypes: int64(5)
memory usage: 91.6 MB


In [5]:
# 문자열 클래스일 경우 숫자로 변환
le = LabelEncoder()
y = le.fit_transform(y)

In [6]:
# 데이터 분리하기
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
print(X_train.shape, X_val.shape, y_val.shape, y_train.shape)

(1920000, 5) (480000, 5) (480000,) (1920000,)


In [ ]:
XGBClassifier(tree_method="hist", device="cuda")

In [ ]:
# 학습
model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    # use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method="hist", 
    device="cuda"
)

model.fit(X_train, y_train)
y_val_train = model.predict(X_val)

In [13]:
# 교차검증
f1_micro = make_scorer(f1_score, average='micro')
scores = cross_val_score(model, X_train, y_train, cv=5, scoring=f1_micro)

In [9]:
print("교차검증 평균 F1:", scores.mean())

교차검증 평균 F1: 0.8351734375


In [15]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300]
}

grid = GridSearchCV(
    estimator=XGBClassifier(objective='multi:softmax', num_class=5, eval_metric='mlogloss'),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3
)

grid.fit(X_train, y_train)

print("최적 F1:", grid.best_score_)
print("최적 파라미터:", grid.best_params_)

최적 F1: 0.36283288590914387
최적 파라미터: {'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 300}


In [16]:
y_val_pred = model.predict(X_val)
print("최종 F1 (micro):", f1_score(y_val, y_val_pred, average='micro'))
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

최종 F1 (micro): 0.8356416666666666
              precision    recall  f1-score   support

           A       0.43      0.02      0.03       168
           B       0.00      0.00      0.00        29
           C       0.57      0.34      0.42     25649
           D       0.52      0.30      0.38     69483
           E       0.87      0.97      0.92    384671

    accuracy                           0.84    480000
   macro avg       0.48      0.32      0.35    480000
weighted avg       0.81      0.84      0.81    480000

